# Deep Dive: TD Learning, n-step Returns, Advantages, and GAE

## Introduction

Welcome to this comprehensive guide on key reinforcement learning concepts! This notebook is designed for learners who understand the basics of policy gradients and REINFORCE, but want to deeply understand more sophisticated methods for estimating returns and advantages.

### What We'll Cover

1. **TD (Temporal Difference) Learning**: How to learn value functions from incomplete episodes
2. **n-step Returns**: Balancing between immediate and long-term rewards
3. **Advantage Functions**: Measuring how much better an action is than average
4. **Generalized Advantage Estimation (GAE)**: A practical method that combines the benefits of all the above

### Why These Matter

In REINFORCE, we use the full return (sum of all future rewards) to estimate the gradient. While this is unbiased, it has **high variance** - returns can vary wildly even for the same state and action. This makes learning slow and unstable.

The techniques we'll learn today help us:
- **Reduce variance** in our estimates
- **Learn from incomplete episodes** (don't need to wait for the end)
- **Balance bias and variance** for faster, more stable learning

Let's dive in!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch
from matplotlib.colors import LinearSegmentedColormap
from typing import List, Tuple, Dict
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Setup complete!")

In [ ]:
# ============================================================================
# GRIDWORLD ENVIRONMENTS AND VISUALIZATION UTILITIES
# ============================================================================

class CorridorWorld:
    """
    Simple 1D corridor environment: S -> -> -> -> G
    Perfect for understanding step-by-step updates.
    """
    def __init__(self, length=5):
        self.length = length
        self.start = 0
        self.goal = length - 1
        self.states = list(range(length))
        
    def reset(self):
        return self.start
    
    def step(self, state):
        """Take one step right (deterministic policy)."""
        if state == self.goal:
            return state, 0, True  # stay at goal, no reward, done
        
        next_state = state + 1
        reward = 10.0 if next_state == self.goal else -1.0
        done = next_state == self.goal
        return next_state, reward, done
    
    def generate_episode(self):
        """Generate a complete episode."""
        states, rewards = [self.start], []
        state = self.start
        
        while state != self.goal:
            next_state, reward, done = self.step(state)
            states.append(next_state)
            rewards.append(reward)
            state = next_state
        
        return states, rewards


class GridWorld:
    """
    2D GridWorld environment with obstacles.
    
    Layout (4x4 example):
        S  .  .  .
        .  X  X  .
        .  .  .  .
        .  .  .  G
    """
    def __init__(self, size=(4, 4), obstacles=None, goal=None):
        self.size = size
        self.obstacles = obstacles or [(1, 1), (1, 2)]
        self.start = (0, 0)
        self.goal = goal or (size[0]-1, size[1]-1)
        
        # Create state index mapping
        self.state_to_idx = {}
        self.idx_to_state = {}
        idx = 0
        for i in range(size[0]):
            for j in range(size[1]):
                if (i, j) not in self.obstacles:
                    self.state_to_idx[(i, j)] = idx
                    self.idx_to_state[idx] = (i, j)
                    idx += 1
        
        self.n_states = len(self.state_to_idx)
    
    def reset(self):
        return self.start
    
    def get_next_state(self, state, action):
        """Get next state given action (0=up, 1=right, 2=down, 3=left)."""
        i, j = state
        moves = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # up, right, down, left
        di, dj = moves[action]
        next_i, next_j = i + di, j + dj
        
        # Check bounds and obstacles
        if (0 <= next_i < self.size[0] and 0 <= next_j < self.size[1] and
            (next_i, next_j) not in self.obstacles):
            return (next_i, next_j)
        return state  # stay in place if invalid
    
    def step(self, state, action):
        """Take a step in the environment."""
        next_state = self.get_next_state(state, action)
        
        if next_state == self.goal:
            reward = 10.0
            done = True
        else:
            reward = -1.0
            done = False
        
        return next_state, reward, done


def plot_corridor_values(values, title="Corridor State Values", 
                        trajectory=None, highlight_states=None):
    """
    Visualize values for corridor environment.
    
    Args:
        values: Dict or list of state values
        title: Plot title
        trajectory: Optional list of states to highlight
        highlight_states: Optional list of states to highlight differently
    """
    if isinstance(values, dict):
        n_states = len(values)
        value_array = np.array([values.get(i, 0) for i in range(n_states)])
    else:
        value_array = np.array(values)
        n_states = len(value_array)
    
    fig, ax = plt.subplots(figsize=(12, 3))
    
    # Create color map
    vmin, vmax = value_array.min(), value_array.max()
    if vmax == vmin:
        vmax = vmin + 1
    
    # Draw states
    for i in range(n_states):
        # Determine color
        norm_val = (value_array[i] - vmin) / (vmax - vmin)
        color = plt.cm.RdYlGn(norm_val)
        
        # Draw rectangle
        rect = Rectangle((i, 0), 1, 1, linewidth=2, 
                        edgecolor='black', facecolor=color)
        ax.add_patch(rect)
        
        # Add value text
        ax.text(i + 0.5, 0.5, f'{value_array[i]:.2f}', 
               ha='center', va='center', fontsize=12, fontweight='bold')
        
        # Add state label
        label = 'S' if i == 0 else ('G' if i == n_states-1 else f's{i}')
        ax.text(i + 0.5, -0.3, label, ha='center', va='top', fontsize=10)
    
    # Highlight trajectory if provided
    if trajectory:
        for idx in range(len(trajectory) - 1):
            start = trajectory[idx]
            end = trajectory[idx + 1]
            arrow = FancyArrowPatch((start + 0.5, 1.3), (end + 0.5, 1.3),
                                  arrowstyle='->', mutation_scale=20, 
                                  linewidth=2, color='blue')
            ax.add_patch(arrow)
    
    ax.set_xlim(-0.5, n_states + 0.5)
    ax.set_ylim(-0.5, 1.8)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    return fig, ax


def plot_gridworld_values(env, values, title="GridWorld State Values",
                         show_numbers=True, trajectory=None):
    """
    Visualize values for 2D gridworld as a heatmap.
    
    Args:
        env: GridWorld environment
        values: Dict mapping state tuples to values
        title: Plot title
        show_numbers: Whether to show numerical values
        trajectory: Optional list of states forming a trajectory
    """
    # Create value grid
    value_grid = np.full(env.size, np.nan)
    for state, value in values.items():
        if state not in env.obstacles:
            value_grid[state] = value
    
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Plot heatmap
    im = ax.imshow(value_grid, cmap='RdYlGn', aspect='equal')
    
    # Add grid lines
    for i in range(env.size[0] + 1):
        ax.axhline(i - 0.5, color='black', linewidth=2)
    for j in range(env.size[1] + 1):
        ax.axvline(j - 0.5, color='black', linewidth=2)
    
    # Add text annotations
    for i in range(env.size[0]):
        for j in range(env.size[1]):
            if (i, j) in env.obstacles:
                ax.add_patch(Rectangle((j-0.5, i-0.5), 1, 1, 
                                      fill=True, color='gray', zorder=2))
                ax.text(j, i, 'X', ha='center', va='center',
                       fontsize=20, fontweight='bold', color='white')
            elif (i, j) == env.start:
                ax.text(j, i-0.35, 'START', ha='center', va='center',
                       fontsize=9, fontweight='bold', color='blue')
                if show_numbers and not np.isnan(value_grid[i, j]):
                    ax.text(j, i+0.2, f'{value_grid[i, j]:.1f}',
                           ha='center', va='center', fontsize=10)
            elif (i, j) == env.goal:
                ax.text(j, i-0.35, 'GOAL', ha='center', va='center',
                       fontsize=9, fontweight='bold', color='green')
                if show_numbers and not np.isnan(value_grid[i, j]):
                    ax.text(j, i+0.2, f'{value_grid[i, j]:.1f}',
                           ha='center', va='center', fontsize=10)
            elif show_numbers and not np.isnan(value_grid[i, j]):
                ax.text(j, i, f'{value_grid[i, j]:.1f}',
                       ha='center', va='center', fontsize=11, fontweight='bold')
    
    # Draw trajectory if provided
    if trajectory:
        for idx in range(len(trajectory) - 1):
            i1, j1 = trajectory[idx]
            i2, j2 = trajectory[idx + 1]
            ax.annotate('', xy=(j2, i2), xytext=(j1, i1),
                       arrowprops=dict(arrowstyle='->', lw=3, color='blue',
                                     alpha=0.7))
    
    ax.set_xticks(range(env.size[1]))
    ax.set_yticks(range(env.size[0]))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('State Value', fontsize=11)
    
    plt.tight_layout()
    return fig, ax


def plot_value_evolution(value_history, env_type='corridor', title="Value Evolution"):
    """
    Plot how values evolve over episodes/iterations.
    
    Args:
        value_history: List of value dicts, one per iteration
        env_type: 'corridor' or 'gridworld'
        title: Plot title
    """
    n_iterations = len(value_history)
    
    if env_type == 'corridor':
        # Extract values for each state over time
        n_states = len(value_history[0])
        state_values = np.zeros((n_iterations, n_states))
        
        for t, values in enumerate(value_history):
            for s in range(n_states):
                state_values[t, s] = values.get(s, 0)
        
        # Plot evolution
        fig, ax = plt.subplots(figsize=(10, 6))
        for s in range(n_states):
            label = 'Start' if s == 0 else ('Goal' if s == n_states-1 else f'State {s}')
            ax.plot(state_values[:, s], marker='o', label=label, linewidth=2)
        
        ax.set_xlabel('Episode', fontsize=12)
        ax.set_ylabel('State Value', fontsize=12)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.legend(fontsize=10, loc='best')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
    
    return fig, ax

print("GridWorld environments and visualization utilities loaded!")

---

# 1. Temporal Difference (TD) Learning

## 1.1 High-Level Intuition

Imagine you're learning to predict how long your commute will take. You could:

**Option A (Monte Carlo)**: Wait until you arrive home, then update your prediction based on the actual total time.

**Option B (TD Learning)**: While commuting, continuously update your prediction. If you expected to reach a checkpoint in 10 minutes but it took 12, adjust your prediction *right now* rather than waiting.

TD learning is Option B. It's called "Temporal Difference" because we use the **difference** between predictions at different **time steps**.

### Key Insight

Instead of waiting for the full return $G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + ...$, we can use:
- The immediate reward $r_t$ (which we observe)
- Our *current estimate* of the value of the next state $V(s_{t+1})$

This creates a **bootstrapped estimate**: we use our own predictions to improve our predictions!

---

## 1.2 Formal Definitions

Let's define our terms precisely.

### Value Function

The **state value function** $V^\pi(s)$ under policy $\pi$ is the expected cumulative discounted reward starting from state $s$:

$$V^\pi(s) = \mathbb{E}_\pi\left[G_t \mid s_t = s\right] = \mathbb{E}_\pi\left[\sum_{k=0}^{\infty} \gamma^k r_{t+k} \mid s_t = s\right]$$

Where:
- $V^\pi(s)$ = value of state $s$ under policy $\pi$
- $G_t$ = return (cumulative discounted reward) from time $t$
- $\gamma \in [0,1]$ = discount factor (importance of future rewards)
- $r_{t+k}$ = reward received at time $t+k$
- $\mathbb{E}_\pi[\cdot]$ = expectation when following policy $\pi$

**Important**: $V^\pi(s)$ is the **true** value, which is unknown! We'll estimate it.

### The Bellman Equation

The value function satisfies the **Bellman equation**:

$$V^\pi(s) = \mathbb{E}_\pi[r_t + \gamma V^\pi(s_{t+1}) \mid s_t = s]$$

**Derivation (step by step)**:

Starting from the definition:
$$V^\pi(s) = \mathbb{E}_\pi\left[\sum_{k=0}^{\infty} \gamma^k r_{t+k} \mid s_t = s\right]$$

Separate the first reward from the rest:
$$V^\pi(s) = \mathbb{E}_\pi\left[r_t + \sum_{k=1}^{\infty} \gamma^k r_{t+k} \mid s_t = s\right]$$

Factor out $\gamma$ from the sum:
$$V^\pi(s) = \mathbb{E}_\pi\left[r_t + \gamma \sum_{k=1}^{\infty} \gamma^{k-1} r_{t+k} \mid s_t = s\right]$$

Change index: let $j = k-1$, so when $k=1$, $j=0$:
$$V^\pi(s) = \mathbb{E}_\pi\left[r_t + \gamma \sum_{j=0}^{\infty} \gamma^{j} r_{t+1+j} \mid s_t = s\right]$$

The sum $\sum_{j=0}^{\infty} \gamma^{j} r_{t+1+j}$ is just the return starting from time $t+1$, which is $G_{t+1}$.

By definition, $V^\pi(s_{t+1}) = \mathbb{E}_\pi[G_{t+1} \mid s_{t+1}]$, so:
$$V^\pi(s) = \mathbb{E}_\pi[r_t + \gamma V^\pi(s_{t+1}) \mid s_t = s]$$

**Interpretation**: The value of a state equals the expected immediate reward plus the discounted value of the next state.

---

## 1.3 TD(0) Algorithm

Now, how do we **estimate** $V^\pi(s)$ from data? We don't know the true value function or the expectation. We only have **samples**.

### The TD Target

From the Bellman equation, the true value satisfies:
$$V^\pi(s_t) = \mathbb{E}_\pi[r_t + \gamma V^\pi(s_{t+1})]$$

We can't compute the expectation, but we can **sample** it! When we observe a transition $(s_t, r_t, s_{t+1})$, we form the **TD target**:

$$\text{TD Target} = r_t + \gamma V(s_{t+1})$$

Where $V(s_{t+1})$ is our **current estimate** of the value of state $s_{t+1}$.

**Key point**: This is a **sample-based estimate** of the right-hand side of the Bellman equation. The expectation is replaced by a single sample.

### The TD Error

The **TD error** measures how wrong our current estimate is:

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

Where:
- $\delta_t$ = TD error at time $t$
- $r_t$ = observed reward (from environment)
- $V(s_{t+1})$ = our current estimate of next state's value
- $V(s_t)$ = our current estimate of current state's value
- $\gamma$ = discount factor

**Interpretation**: 
- If $\delta_t > 0$: We received more reward (or reached a better state) than expected → increase $V(s_t)$
- If $\delta_t < 0$: We received less reward (or reached a worse state) than expected → decrease $V(s_t)$

### The TD(0) Update Rule

We update our value estimate using:

$$V(s_t) \leftarrow V(s_t) + \alpha \delta_t$$

Or equivalently:

$$V(s_t) \leftarrow V(s_t) + \alpha [r_t + \gamma V(s_{t+1}) - V(s_t)]$$

Where:
- $\alpha \in (0,1]$ = learning rate (step size)

This can be rewritten as:

$$V(s_t) \leftarrow (1-\alpha) V(s_t) + \alpha [r_t + \gamma V(s_{t+1})]$$

**Interpretation**: We move our estimate toward the TD target, with $\alpha$ controlling how big each step is.

---

## 1.4 Numerical Example: TD(0)

Let's work through a concrete example by hand.

### Setup

Consider a simple environment with 4 states: $S = \{s_1, s_2, s_3, s_4\}$

**Parameters**:
- Discount factor: $\gamma = 0.9$
- Learning rate: $\alpha = 0.1$

**Initial value estimates** (our starting guesses):
- $V(s_1) = 0$
- $V(s_2) = 0$
- $V(s_3) = 0$
- $V(s_4) = 0$ (terminal state)

**Trajectory** (what we observe):
$$s_1 \xrightarrow{r=1} s_2 \xrightarrow{r=2} s_3 \xrightarrow{r=3} s_4$$

### Step-by-Step TD(0) Updates

**Time t=0**: At state $s_1$, take action, receive reward $r_0 = 1$, arrive at $s_2$

1. Compute TD target:
   $$\text{TD Target} = r_0 + \gamma V(s_2) = 1 + 0.9 \times 0 = 1.0$$

2. Compute TD error:
   $$\delta_0 = r_0 + \gamma V(s_2) - V(s_1) = 1 + 0.9 \times 0 - 0 = 1.0$$

3. Update $V(s_1)$:
   $$V(s_1) \leftarrow V(s_1) + \alpha \delta_0 = 0 + 0.1 \times 1.0 = 0.1$$

**Current estimates**: $V(s_1) = 0.1, V(s_2) = 0, V(s_3) = 0, V(s_4) = 0$

---

**Time t=1**: At state $s_2$, take action, receive reward $r_1 = 2$, arrive at $s_3$

1. Compute TD target:
   $$\text{TD Target} = r_1 + \gamma V(s_3) = 2 + 0.9 \times 0 = 2.0$$

2. Compute TD error:
   $$\delta_1 = r_1 + \gamma V(s_3) - V(s_2) = 2 + 0.9 \times 0 - 0 = 2.0$$

3. Update $V(s_2)$:
   $$V(s_2) \leftarrow V(s_2) + \alpha \delta_1 = 0 + 0.1 \times 2.0 = 0.2$$

**Current estimates**: $V(s_1) = 0.1, V(s_2) = 0.2, V(s_3) = 0, V(s_4) = 0$

---

**Time t=2**: At state $s_3$, take action, receive reward $r_2 = 3$, arrive at $s_4$ (terminal)

1. Compute TD target:
   $$\text{TD Target} = r_2 + \gamma V(s_4) = 3 + 0.9 \times 0 = 3.0$$
   
   (Terminal states have value 0 by convention)

2. Compute TD error:
   $$\delta_2 = r_2 + \gamma V(s_4) - V(s_3) = 3 + 0.9 \times 0 - 0 = 3.0$$

3. Update $V(s_3)$:
   $$V(s_3) \leftarrow V(s_3) + \alpha \delta_2 = 0 + 0.1 \times 3.0 = 0.3$$

**Final estimates after one episode**: $V(s_1) = 0.1, V(s_2) = 0.2, V(s_3) = 0.3, V(s_4) = 0$

### Verification: What are the True Values?

For this specific trajectory, the actual returns would be:
- From $s_1$: $G_0 = 1 + 0.9 \times 2 + 0.9^2 \times 3 = 1 + 1.8 + 2.43 = 5.23$
- From $s_2$: $G_1 = 2 + 0.9 \times 3 = 2 + 2.7 = 4.7$
- From $s_3$: $G_2 = 3$

Our estimates ($0.1, 0.2, 0.3$) are much lower than the true values! This is expected - TD learning requires **many episodes** to converge. With each episode, the estimates get better.

---

## 1.5 Code Implementation: TD(0)

In [ ]:
def td_zero_update(V: dict, state: int, reward: float, next_state: int, 
                   gamma: float, alpha: float, is_terminal: bool = False) -> float:
    """
    Perform a single TD(0) update.
    
    Args:
        V: Dictionary mapping states to their current value estimates
        state: Current state
        reward: Reward received
        next_state: Next state
        gamma: Discount factor
        alpha: Learning rate
        is_terminal: Whether next_state is terminal
    
    Returns:
        TD error (for monitoring)
    """
    # Get current value estimates
    v_current = V.get(state, 0.0)
    v_next = 0.0 if is_terminal else V.get(next_state, 0.0)
    
    # Compute TD target
    td_target = reward + gamma * v_next
    
    # Compute TD error
    td_error = td_target - v_current
    
    # Update value
    V[state] = v_current + alpha * td_error
    
    return td_error

# Reproduce our numerical example
print("=" * 60)
print("TD(0) Numerical Example")
print("=" * 60)

# Initialize
V = {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}
gamma = 0.9
alpha = 0.1

# Trajectory: s1 --(r=1)--> s2 --(r=2)--> s3 --(r=3)--> s4
trajectory = [(1, 1, 2, False), (2, 2, 3, False), (3, 3, 4, True)]

print(f"\nInitial values: {V}")
print(f"Parameters: gamma={gamma}, alpha={alpha}\n")

for t, (state, reward, next_state, is_terminal) in enumerate(trajectory):
    print(f"--- Time t={t} ---")
    print(f"Transition: s{state} --(r={reward})--> s{next_state}")
    
    v_before = V[state]
    v_next = 0.0 if is_terminal else V[next_state]
    
    print(f"Before update: V(s{state}) = {v_before:.1f}")
    print(f"Next state value: V(s{next_state}) = {v_next:.1f}")
    
    td_error = td_zero_update(V, state, reward, next_state, gamma, alpha, is_terminal)
    
    td_target = reward + gamma * v_next
    print(f"TD Target: {reward} + {gamma} × {v_next} = {td_target:.1f}")
    print(f"TD Error: {td_target:.1f} - {v_before:.1f} = {td_error:.1f}")
    print(f"After update: V(s{state}) = {V[state]:.1f}")
    print()

print(f"Final value estimates: {V}")
print("\nNote: These will converge to true values with more episodes!")

### Running TD(0) for Multiple Episodes

In [ ]:
# Simple environment: 4-state chain
# s1 -> s2 -> s3 -> s4 (terminal)
# Rewards: 1, 2, 3

def run_td_learning(num_episodes: int, gamma: float = 0.9, alpha: float = 0.1):
    """
    Run TD(0) for multiple episodes on a simple chain environment.
    """
    # Initialize value function
    V = {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}
    
    # Fixed trajectory for deterministic environment
    trajectory = [(1, 1, 2, False), (2, 2, 3, False), (3, 3, 4, True)]
    
    # Track value estimates over time
    history = {s: [] for s in [1, 2, 3]}
    
    for episode in range(num_episodes):
        for state, reward, next_state, is_terminal in trajectory:
            td_zero_update(V, state, reward, next_state, gamma, alpha, is_terminal)
        
        # Record current estimates
        for s in [1, 2, 3]:
            history[s].append(V[s])
    
    return V, history

# Run TD learning
V_final, history = run_td_learning(num_episodes=100, gamma=0.9, alpha=0.1)

# Plot convergence
plt.figure(figsize=(10, 6))
for state in [1, 2, 3]:
    plt.plot(history[state], label=f'V(s{state})', linewidth=2)

# True values (computed analytically)
true_values = {1: 5.23, 2: 4.7, 3: 3.0}
for state in [1, 2, 3]:
    plt.axhline(y=true_values[state], color='gray', linestyle='--', alpha=0.5)

plt.xlabel('Episode', fontsize=12)
plt.ylabel('Value Estimate', fontsize=12)
plt.title('TD(0) Convergence', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nFinal value estimates after 100 episodes:")
for state in [1, 2, 3]:
    print(f"V(s{state}) = {V_final[state]:.2f} (true value ≈ {true_values[state]:.2f})")

## 1.6 Visual Example: TD Learning in Corridor World

Let's see TD learning in action with our corridor environment! We'll watch how the value estimates evolve over multiple episodes.


In [ ]:
# Create corridor environment
print("="*70)
print("TD LEARNING VISUAL EXAMPLE: Corridor World")
print("="*70)

env = CorridorWorld(length=5)
print(f"\nEnvironment: {env.length} states")
print(f"Start: State {env.start}, Goal: State {env.goal}")
print(f"Rewards: -1 per step, +10 for reaching goal\n")

# Show initial corridor
states, rewards = env.generate_episode()
print(f"Example trajectory: {' -> '.join([f's{s}' for s in states])}")
print(f"Rewards: {rewards}\n")

# Initialize value function
V_corridor = {s: 0.0 for s in env.states}
gamma = 0.9
alpha = 0.1

# Track value history
value_history = [V_corridor.copy()]
episodes_to_show = [0, 1, 5, 10, 25, 50]
n_episodes = 50

print("Training TD(0) for 50 episodes...\n")

# Run TD learning
for episode in range(1, n_episodes + 1):
    state = env.reset()
    
    while True:
        next_state, reward, done = env.step(state)
        
        # TD update
        v_current = V_corridor[state]
        v_next = V_corridor[next_state]
        td_error = reward + gamma * v_next - v_current
        V_corridor[state] = v_current + alpha * td_error
        
        if done:
            break
        state = next_state
    
    value_history.append(V_corridor.copy())

print("Training complete!\n")

# Visualize value evolution at key episodes
fig = plt.figure(figsize=(15, 10))

for idx, ep in enumerate(episodes_to_show):
    ax = plt.subplot(2, 3, idx + 1)
    values = value_history[ep]
    
    # Create visualization
    value_array = np.array([values[s] for s in range(env.length)])
    vmin, vmax = -5, 10
    
    for i in range(env.length):
        norm_val = (value_array[i] - vmin) / (vmax - vmin)
        color = plt.cm.RdYlGn(norm_val)
        
        rect = Rectangle((i, 0), 1, 1, linewidth=2, 
                        edgecolor='black', facecolor=color)
        ax.add_patch(rect)
        
        ax.text(i + 0.5, 0.5, f'{value_array[i]:.1f}', 
               ha='center', va='center', fontsize=11, fontweight='bold')
        
        label = 'S' if i == 0 else ('G' if i == env.length-1 else f's{i}')
        ax.text(i + 0.5, -0.25, label, ha='center', va='top', fontsize=9)
    
    ax.set_xlim(-0.5, env.length + 0.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(f'Episode {ep}', fontsize=12, fontweight='bold')

plt.suptitle('TD(0) Learning: Value Estimates Over Time', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# Plot convergence curves
plot_value_evolution(value_history[:51], env_type='corridor', 
                     title='TD(0) Value Convergence in Corridor World')
plt.show()

print("\nFinal Value Estimates:")
print("-" * 40)
for s in env.states:
    label = 'Start' if s == 0 else ('Goal' if s == env.goal else f'State {s}')
    print(f"{label:10}: V(s) = {V_corridor[s]:.2f}")


## 1.7 Visual Example: TD Learning in 2D GridWorld

Now let's see TD learning in a more realistic 2D environment with obstacles! This shows how values propagate from the goal state backward through the grid.


In [ ]:
# 2D GridWorld TD Learning Example
print("="*70)
print("TD LEARNING IN 2D GRIDWORLD")
print("="*70)

# Create 2D gridworld
grid_env = GridWorld(size=(4, 4), obstacles=[(1, 1), (1, 2)])

print("\nGridWorld Layout:")
print("  S  .  .  .")
print("  .  X  X  .")
print("  .  .  .  .")
print("  .  .  .  G")
print("\nS = Start, G = Goal, X = Obstacle, . = Empty")
print(f"Rewards: -1 per step, +10 for reaching goal\n")

# Define a simple policy (move towards goal)
def simple_policy(state, goal):
    """Simple policy: move towards goal (right or down)."""
    i, j = state
    gi, gj = goal
    
    # Prefer right if not at right edge, else down
    if j < gj:
        return 1  # right
    elif i < gi:
        return 2  # down
    elif i > gi:
        return 0  # up
    else:
        return 3  # left

# Initialize value function for all non-obstacle states
V_grid = {}
for i in range(grid_env.size[0]):
    for j in range(grid_env.size[1]):
        if (i, j) not in grid_env.obstacles:
            V_grid[(i, j)] = 0.0

gamma = 0.9
alpha = 0.1
n_episodes_grid = 100

# Track value history for visualization
grid_value_history = []
episodes_to_viz = [0, 5, 15, 30, 60, 100]

print(f"Training TD(0) for {n_episodes_grid} episodes...\n")

# Run TD learning
for episode in range(n_episodes_grid + 1):
    if episode in episodes_to_viz:
        grid_value_history.append(V_grid.copy())
    
    if episode == n_episodes_grid:
        break
        
    state = grid_env.reset()
    steps = 0
    max_steps = 50  # Prevent infinite loops
    
    while steps < max_steps:
        # Take action using simple policy
        action = simple_policy(state, grid_env.goal)
        next_state, reward, done = grid_env.step(state, action)
        
        # TD update
        v_current = V_grid[state]
        v_next = V_grid[next_state]
        td_error = reward + gamma * v_next - v_current
        V_grid[state] = v_current + alpha * td_error
        
        if done:
            break
        
        state = next_state
        steps += 1

print("Training complete!\n")

# Visualize evolution
fig = plt.figure(figsize=(18, 12))

for idx, ep in enumerate(episodes_to_viz):
    ax = plt.subplot(2, 3, idx + 1)
    
    values = grid_value_history[idx]
    value_grid = np.full(grid_env.size, np.nan)
    
    for state, value in values.items():
        if state not in grid_env.obstacles:
            value_grid[state] = value
    
    # Plot heatmap
    im = ax.imshow(value_grid, cmap='RdYlGn', aspect='equal', vmin=-5, vmax=10)
    
    # Add grid lines
    for i in range(grid_env.size[0] + 1):
        ax.axhline(i - 0.5, color='black', linewidth=2)
    for j in range(grid_env.size[1] + 1):
        ax.axvline(j - 0.5, color='black', linewidth=2)
    
    # Add annotations
    for i in range(grid_env.size[0]):
        for j in range(grid_env.size[1]):
            if (i, j) in grid_env.obstacles:
                ax.add_patch(Rectangle((j-0.5, i-0.5), 1, 1, 
                                      fill=True, color='gray', zorder=2))
                ax.text(j, i, 'X', ha='center', va='center',
                       fontsize=16, fontweight='bold', color='white')
            elif (i, j) == grid_env.start:
                ax.text(j, i-0.3, 'S', ha='center', va='center',
                       fontsize=10, fontweight='bold', color='blue')
                if not np.isnan(value_grid[i, j]):
                    ax.text(j, i+0.2, f'{value_grid[i, j]:.1f}',
                           ha='center', va='center', fontsize=9)
            elif (i, j) == grid_env.goal:
                ax.text(j, i-0.3, 'G', ha='center', va='center',
                       fontsize=10, fontweight='bold', color='green')
                if not np.isnan(value_grid[i, j]):
                    ax.text(j, i+0.2, f'{value_grid[i, j]:.1f}',
                           ha='center', va='center', fontsize=9)
            elif not np.isnan(value_grid[i, j]):
                ax.text(j, i, f'{value_grid[i, j]:.1f}',
                       ha='center', va='center', fontsize=10, fontweight='bold')
    
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'Episode {ep}', fontsize=12, fontweight='bold')

plt.suptitle('TD(0) Learning in 2D GridWorld: Value Propagation from Goal', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# Show final values
print("\nFinal Value Function:")
print("="*40)
plot_gridworld_values(grid_env, V_grid, 
                      title="Final Value Function After TD(0) Training",
                      show_numbers=True)
plt.show()

print("\nKey Observations:")
print("-" * 60)
print("1. Values are highest at the goal (10.0)")
print("2. Values decrease with distance from goal")
print("3. The obstacles create a 'valley' in the value landscape")
print("4. States closer to the goal learn faster (updated more frequently)")
print("5. TD learning propagates values backward from the goal!")


### Key Observations

1. **TD learning updates online**: We update after each step, not waiting for the episode to end
2. **Bootstrapping**: We use our own estimates ($V(s_{t+1})$) to update other estimates ($V(s_t)$)
3. **Convergence**: With enough episodes, TD(0) converges to the true value function
4. **Lower variance than Monte Carlo**: By using bootstrapping, TD has lower variance but introduces bias (initially)

---

# 2. n-step Returns

## 2.1 High-Level Intuition

TD(0) and Monte Carlo represent two extremes:

- **TD(0)**: Use 1 real reward, then bootstrap (use estimate for the rest)
  - Low variance, but high bias (if our estimate is wrong)
  - Updates quickly

- **Monte Carlo**: Use ALL real rewards until episode end
  - No bias (we use actual returns)
  - High variance (returns can vary a lot)
  - Must wait for episode to end

**n-step returns** give us the middle ground! We use $n$ real rewards, then bootstrap:

- Use $n$ steps of real rewards: $r_t, r_{t+1}, ..., r_{t+n-1}$
- Then bootstrap with $V(s_{t+n})$

### The Bias-Variance Tradeoff

- **Small $n$ (e.g., $n=1$)**: Low variance, higher bias (like TD)
- **Large $n$ (e.g., $n=\infty$)**: High variance, no bias (like MC)
- **Medium $n$**: Balance between the two!

---

## 2.2 Formal Definitions

### The n-step Return

The **n-step return** starting from time $t$ is:

$$G_t^{(n)} = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + ... + \gamma^{n-1} r_{t+n-1} + \gamma^n V(s_{t+n})$$

More compactly:

$$G_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n})$$

Where:
- $G_t^{(n)}$ = n-step return from time $t$
- $n$ = number of steps to look ahead
- $r_{t+k}$ = reward at time $t+k$ (observed from environment)
- $V(s_{t+n})$ = estimated value of state at time $t+n$ (our current estimate)
- $\gamma$ = discount factor

### Special Cases

**1-step return (TD(0))**:
$$G_t^{(1)} = r_t + \gamma V(s_{t+1})$$

This is exactly the TD target we saw before!

**2-step return**:
$$G_t^{(2)} = r_t + \gamma r_{t+1} + \gamma^2 V(s_{t+2})$$

**3-step return**:
$$G_t^{(3)} = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \gamma^3 V(s_{t+3})$$

**$\infty$-step return (Monte Carlo)**:
$$G_t^{(\infty)} = \sum_{k=0}^{\infty} \gamma^k r_{t+k}$$

This is the full return with no bootstrapping!

### Handling Episode Termination

What if the episode ends before $n$ steps? If the episode terminates at time $T$, and $t+n > T$:

$$G_t^{(n)} = \sum_{k=0}^{T-t-1} \gamma^k r_{t+k}$$

We just use all remaining rewards and don't bootstrap (the terminal state has value 0).

---

## 2.3 Computing n-step Returns from Data

Let's be very explicit about how to compute n-step returns from sample trajectories.

### What Data Do We Have?

After collecting a trajectory, we have:
- States: $s_0, s_1, s_2, ..., s_T$
- Rewards: $r_0, r_1, r_2, ..., r_{T-1}$ (where $r_t$ is received after taking action in $s_t$)
- Value estimates: $V(s_0), V(s_1), ..., V(s_T)$ (our current approximation)

### Algorithm for Computing n-step Return

To compute $G_t^{(n)}$ for a given time $t$:

1. **Initialize** cumulative return: $G = 0$
2. **For** $k = 0$ to $\min(n-1, T-t-1)$:
   - Add discounted reward: $G \leftarrow G + \gamma^k r_{t+k}$
3. **If** $t + n < T$ (episode hasn't ended):
   - Add bootstrapped value: $G \leftarrow G + \gamma^n V(s_{t+n})$
4. **Return** $G$

---

## 2.4 Numerical Example: n-step Returns

Let's compute different n-step returns for the same trajectory.

### Setup

**Trajectory**: Same as before
$$s_1 \xrightarrow{r=1} s_2 \xrightarrow{r=2} s_3 \xrightarrow{r=3} s_4 \text{ (terminal)}$$

**Parameters**:
- $\gamma = 0.9$

**Value estimates** (assume we've trained a bit):
- $V(s_1) = 5.0$
- $V(s_2) = 4.5$
- $V(s_3) = 3.0$
- $V(s_4) = 0.0$ (terminal)

### Computing Returns from State $s_1$ (t=0)

**1-step return**:
$$G_0^{(1)} = r_0 + \gamma V(s_1)$$
$$G_0^{(1)} = 1 + 0.9 \times 4.5 = 1 + 4.05 = 5.05$$

**2-step return**:
$$G_0^{(2)} = r_0 + \gamma r_1 + \gamma^2 V(s_2)$$
$$G_0^{(2)} = 1 + 0.9 \times 2 + 0.9^2 \times 3.0$$
$$G_0^{(2)} = 1 + 1.8 + 0.81 \times 3.0$$
$$G_0^{(2)} = 1 + 1.8 + 2.43 = 5.23$$

**3-step return** (reaches terminal state):
$$G_0^{(3)} = r_0 + \gamma r_1 + \gamma^2 r_2 + \gamma^3 V(s_3)$$

But $s_4$ is terminal, so $V(s_4) = 0$:
$$G_0^{(3)} = 1 + 0.9 \times 2 + 0.9^2 \times 3 + 0.9^3 \times 0$$
$$G_0^{(3)} = 1 + 1.8 + 2.43 + 0 = 5.23$$

**Full return (Monte Carlo)**: Same as 3-step since episode ends!
$$G_0^{(\infty)} = 1 + 0.9 \times 2 + 0.9^2 \times 3 = 5.23$$

### Computing Returns from State $s_2$ (t=1)

**1-step return**:
$$G_1^{(1)} = r_1 + \gamma V(s_2) = 2 + 0.9 \times 3.0 = 2 + 2.7 = 4.7$$

**2-step return** (reaches terminal):
$$G_1^{(2)} = r_1 + \gamma r_2 + \gamma^2 V(s_3)$$
$$G_1^{(2)} = 2 + 0.9 \times 3 + 0.9^2 \times 0 = 2 + 2.7 + 0 = 4.7$$

### Summary Table

| Start State | 1-step | 2-step | 3-step | Full (MC) |
|-------------|--------|--------|--------|----------|
| $s_1$ | 5.05 | 5.23 | 5.23 | 5.23 |
| $s_2$ | 4.70 | 4.70 | 4.70 | 4.70 |
| $s_3$ | 3.00 | 3.00 | 3.00 | 3.00 |

**Key observations**:
1. Different $n$ values give different estimates
2. Once we hit the terminal state, all longer returns are identical
3. The 1-step return from $s_1$ (5.05) is closest to our current estimate $V(s_1) = 5.0$
4. The Monte Carlo return (5.23) uses no estimates, only real rewards

---

## 2.5 Code Implementation: n-step Returns

In [ ]:
def compute_n_step_return(rewards: List[float], 
                         states: List[int],
                         V: dict,
                         t: int,
                         n: int,
                         gamma: float) -> float:
    """
    Compute n-step return starting from time t.
    
    Args:
        rewards: List of rewards [r_0, r_1, ...]
        states: List of states [s_0, s_1, ...]
        V: Value function dictionary
        t: Starting time step
        n: Number of steps
        gamma: Discount factor
    
    Returns:
        n-step return G_t^(n)
    """
    T = len(rewards)  # Episode length
    
    # Accumulate rewards
    G = 0.0
    for k in range(min(n, T - t)):
        G += (gamma ** k) * rewards[t + k]
    
    # Add bootstrapped value if episode hasn't ended
    if t + n < len(states):
        G += (gamma ** n) * V.get(states[t + n], 0.0)
    
    return G

# Reproduce numerical example
print("=" * 60)
print("n-step Returns Numerical Example")
print("=" * 60)

# Trajectory data
states = [1, 2, 3, 4]  # s1 -> s2 -> s3 -> s4 (terminal)
rewards = [1, 2, 3]    # r_0=1, r_1=2, r_2=3

# Value estimates
V = {1: 5.0, 2: 4.5, 3: 3.0, 4: 0.0}
gamma = 0.9

print(f"\nTrajectory: s1 --(r=1)--> s2 --(r=2)--> s3 --(r=3)--> s4")
print(f"Value estimates: V(s1)={V[1]}, V(s2)={V[2]}, V(s3)={V[3]}, V(s4)={V[4]}")
print(f"Discount factor: γ={gamma}\n")

# Compute different n-step returns from each state
for start_state in [1, 2, 3]:
    t = start_state - 1  # Convert to 0-indexed time
    print(f"\n--- Returns from state s{start_state} (t={t}) ---")
    
    for n in [1, 2, 3, 100]:  # 100 to simulate infinity
        G_n = compute_n_step_return(rewards, states, V, t, n, gamma)
        n_label = "∞ (MC)" if n == 100 else n
        print(f"G_{t}^({n_label:>6}) = {G_n:.2f}")

print("\n" + "="*60)

# Detailed computation for G_0^(2) from s1
print("\nDetailed computation: G_0^(2) from s1")
print("="*60)
t = 0
n = 2
print(f"Formula: G_0^(2) = r_0 + γ*r_1 + γ²*V(s_2)")
print(f"       = {rewards[0]} + {gamma}×{rewards[1]} + {gamma}²×{V[3]}")
term1 = rewards[0]
term2 = gamma * rewards[1]
term3 = (gamma ** 2) * V[3]
print(f"       = {term1} + {term2:.2f} + {term3:.2f}")
print(f"       = {term1 + term2 + term3:.2f}")

### Visualizing n-step Returns

In [ ]:
# Compute returns for different n values from s1
n_values = list(range(1, 11))
returns_from_s1 = []

for n in n_values:
    G_n = compute_n_step_return(rewards, states, V, t=0, n=n, gamma=0.9)
    returns_from_s1.append(G_n)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(n_values, returns_from_s1, 'bo-', linewidth=2, markersize=8)
plt.axhline(y=5.23, color='red', linestyle='--', linewidth=2, label='Monte Carlo (full return)')
plt.xlabel('n (number of steps)', fontsize=12)
plt.ylabel('n-step Return G₀⁽ⁿ⁾', fontsize=12)
plt.title('n-step Returns from s₁', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("\nObservations:")
print("- At n=1 (TD), return is 5.05 (uses V(s2) estimate)")
print("- At n=2+, return converges to 5.23 (all real rewards used)")
print("- This happens because the episode is only 3 steps long!")

## 2.6 Visual Example: n-step Returns in Corridor World

Let's visualize how different values of n affect our return estimates! We'll show the "lookahead distance" for different n values.


In [ ]:
# n-step Returns Visual Example
print("="*70)
print("N-STEP RETURNS VISUAL EXAMPLE: Comparing Different n Values")
print("="*70)

# Use the same corridor environment
env = CorridorWorld(length=5)

# Generate a trajectory
states_traj, rewards_traj = env.generate_episode()
print(f"\nTrajectory: {' -> '.join([f's{s}' for s in states_traj])}")
print(f"Rewards: {rewards_traj}")

# Let's assume we have some value estimates
V_nstep = {0: 5.0, 1: 6.0, 2: 7.0, 3: 8.5, 4: 0.0}
gamma = 0.9

print(f"\nValue estimates: {V_nstep}")
print(f"Discount factor: γ = {gamma}\n")

# Compute n-step returns for different n from state 0
n_values = [1, 2, 3, 4]
returns_dict = {}

print("Computing n-step returns from Start (s0):")
print("-" * 60)

for n in n_values:
    G_n = compute_n_step_return(rewards_traj, states_traj, V_nstep, t=0, n=n, gamma=gamma)
    returns_dict[n] = G_n
    print(f"n={n}: G_0^({n}) = {G_n:.2f}")

# Visualize the lookahead for different n values
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, n in enumerate(n_values):
    ax = axes[idx]
    
    # Draw the corridor
    for i in range(env.length):
        # Color based on whether this state is used in the n-step return
        if i <= n:  # States involved in n-step return from s0
            color = plt.cm.Blues(0.3 + 0.1 * i)
            alpha = 1.0
        else:
            color = 'lightgray'
            alpha = 0.3
        
        rect = Rectangle((i, 0), 1, 1, linewidth=2, 
                        edgecolor='black', facecolor=color, alpha=alpha)
        ax.add_patch(rect)
        
        # Add value text
        ax.text(i + 0.5, 0.5, f'{V_nstep[i]:.1f}', 
               ha='center', va='center', fontsize=11, fontweight='bold')
        
        label = 'S' if i == 0 else ('G' if i == env.length-1 else f's{i}')
        ax.text(i + 0.5, -0.25, label, ha='center', va='top', fontsize=9)
    
    # Draw arrows showing the lookahead
    for i in range(min(n, len(states_traj)-1)):
        arrow = FancyArrowPatch((i + 0.5, 1.2), (i + 1.5, 1.2),
                              arrowstyle='->', mutation_scale=15, 
                              linewidth=2, color='darkblue')
        ax.add_patch(arrow)
        
        # Add reward label
        ax.text(i + 1, 1.5, f'r={rewards_traj[i]:.0f}', 
               ha='center', va='bottom', fontsize=9, color='darkred')
    
    # Add bootstrap indicator if applicable
    if n < len(states_traj) - 1:
        ax.text(n + 0.5, -0.7, f'Bootstrap\nV(s{n})', 
               ha='center', va='top', fontsize=9, 
               bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
    
    ax.set_xlim(-0.5, env.length + 0.5)
    ax.set_ylim(-1.2, 2.0)
    ax.set_aspect('equal')
    ax.axis('off')
    
    # Title with return value
    title = f'{n}-step Return: G_0^({n}) = {returns_dict[n]:.2f}\n'
    if n == 1:
        title += '(TD - Pure Bootstrap)'
    elif n == len(states_traj) - 1:
        title += '(Monte Carlo - All Real Rewards)'
    else:
        title += f'(Uses {n} real rewards + bootstrap)'
    ax.set_title(title, fontsize=11, fontweight='bold')

plt.suptitle('n-step Returns: Visualizing the Lookahead Distance', 
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# Summary comparison
print("\n" + "="*60)
print("Summary: How n Affects the Return Estimate")
print("="*60)
print(f"{'n':<5} {'Return':<10} {'# Real Rewards':<15} {'Bootstrap?':<12}")
print("-"*60)
for n in n_values:
    uses_bootstrap = 'Yes' if n < len(rewards_traj) else 'No'
    n_rewards = min(n, len(rewards_traj))
    print(f"{n:<5} {returns_dict[n]:<10.2f} {n_rewards:<15} {uses_bootstrap:<12}")

print("\n→ As n increases, we use more real rewards and less bootstrapping")
print("→ This reduces bias but increases variance!")


---

# 3. Advantage Functions

## 3.1 High-Level Intuition

Imagine you're playing basketball. You're about to take a shot. The value function $V(s)$ tells you: "From this position, your team will score an average of 15 points."

But should you take the shot? That depends on whether this action is **better or worse than average**!

- If passing to a teammate would lead to 18 points (better than 15) → advantage is positive → you should pass
- If taking the shot yourself leads to only 12 points (worse than 15) → advantage is negative → don't shoot

The **advantage function** measures: "How much better is this specific action compared to what we'd typically do?"

### Why Advantages Matter for Policy Gradients

In REINFORCE, we use the return $G_t$ to weight the gradient. But this has problems:

1. **High variance**: Returns vary a lot
2. **All positive rewards**: If all returns are positive, we push probability up for *all* actions, even bad ones

**Advantages solve this!** By using $A(s,a)$ instead of $G_t$:
- Positive $A(s,a)$ → this action is better than average → increase its probability
- Negative $A(s,a)$ → this action is worse than average → decrease its probability
- Zero $A(s,a)$ → this action is average → no update

This centers our updates around zero, reducing variance!

---

## 3.2 Formal Definitions

### Q-function (Action-Value Function)

First, we need the **action-value function** $Q^\pi(s,a)$, which is the expected return starting from state $s$, taking action $a$, then following policy $\pi$:

$$Q^\pi(s,a) = \mathbb{E}_\pi[G_t \mid s_t = s, a_t = a]$$

Where:
- $Q^\pi(s,a)$ = expected return from state $s$ after taking action $a$
- $\mathbb{E}_\pi[\cdot]$ = expectation under policy $\pi$

**Interpretation**: $Q^\pi(s,a)$ tells us how good it is to take action $a$ in state $s$.

### Relationship Between V and Q

The value function is the expected Q-value over actions:

$$V^\pi(s) = \mathbb{E}_{a \sim \pi}[Q^\pi(s,a)] = \sum_a \pi(a|s) Q^\pi(s,a)$$

**Derivation**:
$$V^\pi(s) = \mathbb{E}_\pi[G_t \mid s_t = s]$$
$$= \sum_a \pi(a|s) \mathbb{E}_\pi[G_t \mid s_t = s, a_t = a]$$
$$= \sum_a \pi(a|s) Q^\pi(s,a)$$

**Interpretation**: $V(s)$ is the weighted average of $Q(s,a)$ over all possible actions, weighted by $\pi(a|s)$.

### The Advantage Function

The **advantage function** measures how much better action $a$ is compared to the average action:

$$A^\pi(s,a) = Q^\pi(s,a) - V^\pi(s)$$

Where:
- $A^\pi(s,a)$ = advantage of action $a$ in state $s$
- $Q^\pi(s,a)$ = value of taking action $a$ in state $s$
- $V^\pi(s)$ = average value of state $s$ (baseline)

**Interpretation**:
- $A^\pi(s,a) > 0$: Action $a$ is **better** than average
- $A^\pi(s,a) < 0$: Action $a$ is **worse** than average
- $A^\pi(s,a) = 0$: Action $a$ is exactly average

### Key Properties

**Property 1**: The expected advantage is zero:
$$\mathbb{E}_{a \sim \pi}[A^\pi(s,a)] = 0$$

**Proof**:
$$\mathbb{E}_{a \sim \pi}[A^\pi(s,a)] = \mathbb{E}_{a \sim \pi}[Q^\pi(s,a) - V^\pi(s)]$$
$$= \mathbb{E}_{a \sim \pi}[Q^\pi(s,a)] - V^\pi(s)$$
$$= V^\pi(s) - V^\pi(s) = 0$$

This is why advantages reduce variance!

---

## 3.3 Estimating Advantages from Data

The true advantage $A^\pi(s,a)$ is unknown. We need to **estimate** it from samples.

### Method 1: Monte Carlo Advantage Estimate

If we have a value function estimate $V(s)$ and observe return $G_t$:

$$\hat{A}_t = G_t - V(s_t)$$

**Justification**: 
- $G_t$ is a sample-based estimate of $Q^\pi(s_t, a_t)$
- $V(s_t)$ is our estimate of $V^\pi(s_t)$
- So $G_t - V(s_t)$ estimates $Q^\pi(s_t, a_t) - V^\pi(s_t) = A^\pi(s_t, a_t)$

**Properties**:
- Unbiased (if $V(s)$ is correct)
- High variance (because $G_t$ has high variance)

### Method 2: TD Advantage Estimate (1-step)

Using the TD error:

$$\hat{A}_t = \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

**Justification**: The TD error is an estimate of the advantage!

Let's prove this. The true advantage is:
$$A^\pi(s_t, a_t) = Q^\pi(s_t, a_t) - V^\pi(s_t)$$

From the Bellman equation:
$$Q^\pi(s_t, a_t) = \mathbb{E}[r_t + \gamma V^\pi(s_{t+1})]$$

Therefore:
$$A^\pi(s_t, a_t) = \mathbb{E}[r_t + \gamma V^\pi(s_{t+1})] - V^\pi(s_t)$$
$$= \mathbb{E}[r_t + \gamma V^\pi(s_{t+1}) - V^\pi(s_t)]$$

The TD error $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ is a **sample** of this expectation!

**Properties**:
- Lower variance than Monte Carlo
- Introduces bias (if $V(s)$ is not perfect)

### Method 3: n-step Advantage Estimate

Using n-step returns:

$$\hat{A}_t^{(n)} = G_t^{(n)} - V(s_t)$$

Where $G_t^{(n)}$ is the n-step return we defined earlier.

**Properties**:
- Balances bias and variance
- Larger $n$ → less bias, more variance
- Smaller $n$ → more bias, less variance

---

## 3.4 Numerical Example: Advantage Estimation

Let's compute advantages for our familiar trajectory.

### Setup

**Trajectory**:
$$s_1 \xrightarrow{a_1, r=1} s_2 \xrightarrow{a_2, r=2} s_3 \xrightarrow{a_3, r=3} s_4$$

**Parameters**: $\gamma = 0.9$

**Value estimates**:
- $V(s_1) = 5.0$
- $V(s_2) = 4.5$
- $V(s_3) = 3.0$
- $V(s_4) = 0.0$

### Computing Advantages at Each Step

#### Time t=0 (state $s_1$, action $a_1$)

**Monte Carlo advantage** (using full return):
$$G_0 = 1 + 0.9 \times 2 + 0.9^2 \times 3 = 1 + 1.8 + 2.43 = 5.23$$
$$\hat{A}_0^{MC} = G_0 - V(s_1) = 5.23 - 5.0 = 0.23$$

**1-step TD advantage**:
$$\delta_0 = r_0 + \gamma V(s_2) - V(s_1)$$
$$\delta_0 = 1 + 0.9 \times 4.5 - 5.0$$
$$\delta_0 = 1 + 4.05 - 5.0 = 0.05$$
$$\hat{A}_0^{(1)} = \delta_0 = 0.05$$

**2-step advantage**:
$$G_0^{(2)} = 1 + 0.9 \times 2 + 0.9^2 \times 3.0 = 1 + 1.8 + 2.43 = 5.23$$
$$\hat{A}_0^{(2)} = G_0^{(2)} - V(s_1) = 5.23 - 5.0 = 0.23$$

---

#### Time t=1 (state $s_2$, action $a_2$)

**Monte Carlo advantage**:
$$G_1 = 2 + 0.9 \times 3 = 2 + 2.7 = 4.7$$
$$\hat{A}_1^{MC} = G_1 - V(s_2) = 4.7 - 4.5 = 0.2$$

**1-step TD advantage**:
$$\delta_1 = r_1 + \gamma V(s_3) - V(s_2)$$
$$\delta_1 = 2 + 0.9 \times 3.0 - 4.5$$
$$\delta_1 = 2 + 2.7 - 4.5 = 0.2$$
$$\hat{A}_1^{(1)} = \delta_1 = 0.2$$

---

#### Time t=2 (state $s_3$, action $a_3$)

**Monte Carlo advantage**:
$$G_2 = 3$$
$$\hat{A}_2^{MC} = G_2 - V(s_3) = 3 - 3.0 = 0.0$$

**1-step TD advantage**:
$$\delta_2 = r_2 + \gamma V(s_4) - V(s_3)$$
$$\delta_2 = 3 + 0.9 \times 0 - 3.0 = 0.0$$
$$\hat{A}_2^{(1)} = \delta_2 = 0.0$$

### Summary Table

| Time | State | MC Advantage | 1-step (TD) Advantage | 2-step Advantage |
|------|-------|-------------|---------------------|------------------|
| t=0  | $s_1$ | 0.23 | 0.05 | 0.23 |
| t=1  | $s_2$ | 0.20 | 0.20 | 0.20 |
| t=2  | $s_3$ | 0.00 | 0.00 | 0.00 |

**Observations**:
1. All advantages are **positive** → all actions were better than average
2. The 1-step advantage at t=0 (0.05) is smaller → uses more bootstrapping
3. At t=2, advantage is zero → action was exactly as good as expected

---

## 3.5 Code Implementation: Advantage Estimation

In [ ]:
def compute_advantages_mc(rewards: List[float], 
                         states: List[int], 
                         V: dict, 
                         gamma: float) -> List[float]:
    """
    Compute Monte Carlo advantage estimates for a trajectory.
    
    Args:
        rewards: List of rewards
        states: List of states
        V: Value function estimates
        gamma: Discount factor
    
    Returns:
        List of advantage estimates
    """
    T = len(rewards)
    advantages = []
    
    for t in range(T):
        # Compute full return from time t
        G_t = sum(gamma**k * rewards[t+k] for k in range(T-t))
        
        # Advantage = return - baseline
        A_t = G_t - V.get(states[t], 0.0)
        advantages.append(A_t)
    
    return advantages

def compute_advantages_td(rewards: List[float], 
                         states: List[int], 
                         V: dict, 
                         gamma: float) -> List[float]:
    """
    Compute 1-step TD advantage estimates (TD errors).
    
    Args:
        rewards: List of rewards
        states: List of states
        V: Value function estimates
        gamma: Discount factor
    
    Returns:
        List of TD errors (1-step advantages)
    """
    T = len(rewards)
    advantages = []
    
    for t in range(T):
        v_current = V.get(states[t], 0.0)
        v_next = V.get(states[t+1], 0.0) if t+1 < len(states) else 0.0
        
        # TD error = r_t + gamma*V(s_{t+1}) - V(s_t)
        delta_t = rewards[t] + gamma * v_next - v_current
        advantages.append(delta_t)
    
    return advantages

def compute_advantages_n_step(rewards: List[float], 
                             states: List[int], 
                             V: dict, 
                             gamma: float,
                             n: int) -> List[float]:
    """
    Compute n-step advantage estimates.
    
    Args:
        rewards: List of rewards
        states: List of states
        V: Value function estimates
        gamma: Discount factor
        n: Number of steps
    
    Returns:
        List of n-step advantage estimates
    """
    T = len(rewards)
    advantages = []
    
    for t in range(T):
        # Compute n-step return
        G_t_n = compute_n_step_return(rewards, states, V, t, n, gamma)
        
        # Advantage = n-step return - baseline
        A_t = G_t_n - V.get(states[t], 0.0)
        advantages.append(A_t)
    
    return advantages

# Reproduce numerical example
print("=" * 60)
print("Advantage Estimation Numerical Example")
print("=" * 60)

# Trajectory
states = [1, 2, 3, 4]
rewards = [1, 2, 3]

# Value estimates
V = {1: 5.0, 2: 4.5, 3: 3.0, 4: 0.0}
gamma = 0.9

print(f"\nTrajectory: s1 --(r=1)--> s2 --(r=2)--> s3 --(r=3)--> s4")
print(f"Value estimates: V(s1)={V[1]}, V(s2)={V[2]}, V(s3)={V[3]}, V(s4)={V[4]}")
print(f"Discount factor: γ={gamma}\n")

# Compute advantages using different methods
adv_mc = compute_advantages_mc(rewards, states, V, gamma)
adv_td = compute_advantages_td(rewards, states, V, gamma)
adv_2step = compute_advantages_n_step(rewards, states, V, gamma, n=2)

# Display results
print("\nAdvantage Estimates:")
print("-" * 60)
print(f"{'Time':<6} {'State':<8} {'MC':<10} {'1-step (TD)':<15} {'2-step':<10}")
print("-" * 60)

for t in range(len(rewards)):
    print(f"t={t:<4} s{states[t]:<7} {adv_mc[t]:>6.2f}     {adv_td[t]:>10.2f}      {adv_2step[t]:>6.2f}")

print("\n" + "="*60)

# Detailed computation for t=0, 1-step advantage
print("\nDetailed Computation: 1-step Advantage at t=0")
print("="*60)
t = 0
v_curr = V[states[t]]
v_next = V[states[t+1]]
r = rewards[t]

print(f"Formula: δ_0 = r_0 + γ*V(s_1) - V(s_0)")
print(f"       = {r} + {gamma}×{v_next} - {v_curr}")
td_target = r + gamma * v_next
print(f"       = {r} + {gamma * v_next:.2f} - {v_curr}")
delta = td_target - v_curr
print(f"       = {delta:.2f}")
print(f"\nInterpretation: Action a_0 in state s_1 was slightly better (+0.05) than average.")

### Visualizing Advantage Estimates

In [ ]:
# Compare different advantage estimation methods
time_steps = list(range(len(rewards)))

plt.figure(figsize=(10, 6))
plt.plot(time_steps, adv_mc, 'ro-', label='Monte Carlo', linewidth=2, markersize=10)
plt.plot(time_steps, adv_td, 'bs-', label='1-step (TD)', linewidth=2, markersize=10)
plt.plot(time_steps, adv_2step, 'g^-', label='2-step', linewidth=2, markersize=10)

plt.axhline(y=0, color='black', linestyle='--', alpha=0.3, linewidth=1)
plt.xlabel('Time Step', fontsize=12)
plt.ylabel('Advantage Estimate', fontsize=12)
plt.title('Advantage Estimation Methods Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.xticks(time_steps)
plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("1. All methods give positive advantages (actions were good)")
print("2. 1-step (TD) is different at t=0 due to bootstrapping")
print("3. MC and 2-step agree here (episode is short)")
print("4. Advantage at t=2 is zero (action was exactly as expected)")

---

# 4. Generalized Advantage Estimation (GAE)

## 4.1 High-Level Intuition

We've seen that different n-step advantage estimates have different properties:
- 1-step (TD): Low variance, high bias
- $\infty$-step (MC): No bias, high variance
- n-step: Something in between

**The big question**: Which $n$ should we choose?

**GAE's answer**: Don't choose just one! Use **all** of them with exponentially decaying weights!

### The Core Idea

Instead of picking one value of $n$, GAE computes:
$$\hat{A}_t^{GAE} = (1-\lambda) \times [A_t^{(1)} + \lambda A_t^{(2)} + \lambda^2 A_t^{(3)} + ...]$$

Where:
- $\lambda \in [0,1]$ controls the decay
- Small $\lambda$ → more weight on 1-step (lower variance, more bias)
- Large $\lambda$ → more weight on long-term (less bias, more variance)

This gives us a **single parameter** $\lambda$ to tune the bias-variance tradeoff!

---

## 4.2 Formal Derivation

Let's derive GAE carefully, step by step.

### Starting Point: n-step Advantage

Recall the n-step advantage:
$$\hat{A}_t^{(n)} = G_t^{(n)} - V(s_t)$$

Where:
$$G_t^{(n)} = r_t + \gamma r_{t+1} + ... + \gamma^{n-1} r_{t+n-1} + \gamma^n V(s_{t+n})$$

We can rewrite this by subtracting and adding $V(s_t)$:
$$\hat{A}_t^{(n)} = r_t + \gamma r_{t+1} + ... + \gamma^{n-1} r_{t+n-1} + \gamma^n V(s_{t+n}) - V(s_t)$$

### Expressing n-step Advantage in Terms of TD Errors

Let's work through this for small $n$ first.

**1-step advantage**:
$$\hat{A}_t^{(1)} = r_t + \gamma V(s_{t+1}) - V(s_t) = \delta_t$$

Where $\delta_t$ is the TD error.

**2-step advantage**:
$$\hat{A}_t^{(2)} = r_t + \gamma r_{t+1} + \gamma^2 V(s_{t+2}) - V(s_t)$$

Let's manipulate this:
$$\hat{A}_t^{(2)} = r_t + \gamma V(s_{t+1}) - V(s_t) + \gamma r_{t+1} + \gamma^2 V(s_{t+2}) - \gamma V(s_{t+1})$$
$$= [r_t + \gamma V(s_{t+1}) - V(s_t)] + \gamma[r_{t+1} + \gamma V(s_{t+2}) - V(s_{t+1})]$$
$$= \delta_t + \gamma \delta_{t+1}$$

**3-step advantage**:
Following the same pattern:
$$\hat{A}_t^{(3)} = \delta_t + \gamma \delta_{t+1} + \gamma^2 \delta_{t+2}$$

**General pattern**:
$$\hat{A}_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k \delta_{t+k}$$

### Exponentially-Weighted Average

Now, GAE takes an **exponentially-weighted average** of these n-step advantages:

$$\hat{A}_t^{GAE(\gamma, \lambda)} = (1-\lambda)\sum_{n=1}^{\infty} \lambda^{n-1} \hat{A}_t^{(n)}$$

**Why this weighting?**
- The factor $(1-\lambda)$ normalizes the weights to sum to 1
- The factor $\lambda^{n-1}$ gives exponentially decaying weight to longer horizons

Let's substitute our expression for $\hat{A}_t^{(n)}$:
$$\hat{A}_t^{GAE} = (1-\lambda)\sum_{n=1}^{\infty} \lambda^{n-1} \sum_{k=0}^{n-1} \gamma^k \delta_{t+k}$$

### Simplifying to the GAE Formula

Now comes some algebra. We want to change the order of summation.

For each TD error $\delta_{t+k}$, it appears in all n-step advantages where $n > k$. Specifically:
- $\delta_t$ appears in $\hat{A}_t^{(1)}, \hat{A}_t^{(2)}, \hat{A}_t^{(3)}, ...$ with coefficients $\gamma^0, \gamma^0, \gamma^0, ...$
- $\delta_{t+1}$ appears in $\hat{A}_t^{(2)}, \hat{A}_t^{(3)}, ...$ with coefficients $\gamma^1, \gamma^1, ...$
- $\delta_{t+k}$ appears in $\hat{A}_t^{(k+1)}, \hat{A}_t^{(k+2)}, ...$ with coefficient $\gamma^k$ each time

Changing the order of summation:
$$\hat{A}_t^{GAE} = (1-\lambda)\sum_{k=0}^{\infty} \gamma^k \delta_{t+k} \sum_{n=k+1}^{\infty} \lambda^{n-1}$$

The inner sum is a geometric series. Let $m = n-k-1$:
$$\sum_{n=k+1}^{\infty} \lambda^{n-1} = \sum_{m=0}^{\infty} \lambda^{m+k} = \lambda^k \sum_{m=0}^{\infty} \lambda^m = \frac{\lambda^k}{1-\lambda}$$

Substituting back:
$$\hat{A}_t^{GAE} = (1-\lambda)\sum_{k=0}^{\infty} \gamma^k \delta_{t+k} \cdot \frac{\lambda^k}{1-\lambda}$$

The $(1-\lambda)$ factors cancel:
$$\hat{A}_t^{GAE(\gamma, \lambda)} = \sum_{k=0}^{\infty} (\gamma\lambda)^k \delta_{t+k}$$

**This is the GAE formula!**

---

## 4.3 Understanding the GAE Formula

$$\hat{A}_t^{GAE} = \sum_{k=0}^{\infty} (\gamma\lambda)^k \delta_{t+k}$$

$$= \delta_t + (\gamma\lambda)\delta_{t+1} + (\gamma\lambda)^2\delta_{t+2} + (\gamma\lambda)^3\delta_{t+3} + ...$$

### Interpretation

- We sum **all future TD errors**
- Each TD error is weighted by $(\gamma\lambda)^k$
- Errors further in the future get exponentially less weight
- The parameter $\lambda$ controls how fast the weight decays

### Special Cases

**When $\lambda = 0$**:
$$\hat{A}_t^{GAE} = \delta_t$$
This is just the 1-step TD advantage! (Low variance, high bias)

**When $\lambda = 1$**:
$$\hat{A}_t^{GAE} = \sum_{k=0}^{\infty} \gamma^k \delta_{t+k}$$

Let's see what this equals. Expanding the TD errors:
$$= \sum_{k=0}^{\infty} \gamma^k [r_{t+k} + \gamma V(s_{t+k+1}) - V(s_{t+k})]$$

This is a telescoping sum! The $V$ terms cancel:
$$= \sum_{k=0}^{\infty} \gamma^k r_{t+k} - V(s_t)$$
$$= G_t - V(s_t)$$

This is the Monte Carlo advantage! (No bias, high variance)

**Typical value**: $\lambda \in [0.9, 0.99]$ balances bias and variance.

---

## 4.4 Computing GAE from Data

To compute GAE for a trajectory, we need:

1. **Compute all TD errors** $\delta_t$ for the trajectory
2. **Sum them with exponential weights** $(\gamma\lambda)^k$

### Algorithm

Given trajectory: $(s_0, a_0, r_0), (s_1, a_1, r_1), ..., (s_{T-1}, a_{T-1}, r_{T-1}), s_T$

**Step 1**: Compute all TD errors
```
for t = 0 to T-1:
    δ_t = r_t + γ*V(s_{t+1}) - V(s_t)
```

**Step 2**: Compute GAE (backward pass is more efficient)
```
A_{T-1} = δ_{T-1}
for t = T-2 down to 0:
    A_t = δ_t + (γλ)*A_{t+1}
```

This backward pass is equivalent to:
$$\hat{A}_t = \delta_t + (\gamma\lambda)\delta_{t+1} + (\gamma\lambda)^2\delta_{t+2} + ...$$

---

## 4.5 Numerical Example: GAE

Let's compute GAE for our trajectory with different $\lambda$ values.

### Setup

**Trajectory**: Same as before
$$s_1 \xrightarrow{r=1} s_2 \xrightarrow{r=2} s_3 \xrightarrow{r=3} s_4$$

**Parameters**: $\gamma = 0.9$

**Value estimates**:
- $V(s_1) = 5.0, V(s_2) = 4.5, V(s_3) = 3.0, V(s_4) = 0.0$

### Step 1: Compute TD Errors

We already computed these:
- $\delta_0 = r_0 + \gamma V(s_2) - V(s_1) = 1 + 0.9 \times 4.5 - 5.0 = 0.05$
- $\delta_1 = r_1 + \gamma V(s_3) - V(s_2) = 2 + 0.9 \times 3.0 - 4.5 = 0.2$
- $\delta_2 = r_2 + \gamma V(s_4) - V(s_3) = 3 + 0.9 \times 0 - 3.0 = 0.0$

### Step 2: Compute GAE with $\lambda = 0.95$

**At t=2**:
$$\hat{A}_2^{GAE} = \delta_2 = 0.0$$

**At t=1**:
$$\hat{A}_1^{GAE} = \delta_1 + (\gamma\lambda)\hat{A}_2^{GAE}$$
$$= \delta_1 + (\gamma\lambda) \times 0$$
$$= 0.2 + (0.9 \times 0.95) \times 0.0$$
$$= 0.2 + 0.855 \times 0.0 = 0.2$$

**At t=0**:
$$\hat{A}_0^{GAE} = \delta_0 + (\gamma\lambda)\hat{A}_1^{GAE}$$
$$= \delta_0 + (\gamma\lambda) \times 0.2$$
$$= 0.05 + (0.9 \times 0.95) \times 0.2$$
$$= 0.05 + 0.855 \times 0.2$$
$$= 0.05 + 0.171 = 0.221$$

### Comparing Different $\lambda$ Values

Let's compute GAE for $\lambda \in \{0, 0.5, 0.95, 1.0\}$ at t=0:

**$\lambda = 0$** (1-step TD):
$$\hat{A}_0 = \delta_0 = 0.05$$

**$\lambda = 0.5$**:
$$\hat{A}_0 = \delta_0 + (0.9 \times 0.5)[\delta_1 + (0.9 \times 0.5)\delta_2]$$
$$= 0.05 + 0.45[0.2 + 0.45 \times 0]$$
$$= 0.05 + 0.45 \times 0.2 = 0.05 + 0.09 = 0.14$$

**$\lambda = 0.95$**:
$$\hat{A}_0 = 0.221$$ (computed above)

**$\lambda = 1.0$** (Monte Carlo):
$$\hat{A}_0 = G_0 - V(s_1) = 5.23 - 5.0 = 0.23$$

### Summary

| $\lambda$ | $\hat{A}_0^{GAE}$ | Interpretation |
|-----------|------------------|----------------|
| 0.0 | 0.05 | Pure TD (most bias, least variance) |
| 0.5 | 0.14 | Moderate balance |
| 0.95 | 0.221 | Close to MC |
| 1.0 | 0.23 | Pure MC (no bias, most variance) |

As $\lambda$ increases, GAE incorporates more future rewards and gets closer to the Monte Carlo estimate!

---

## 4.6 Code Implementation: GAE

In [ ]:
def compute_gae(rewards: List[float], 
               states: List[int], 
               V: dict, 
               gamma: float, 
               lambda_: float) -> List[float]:
    """
    Compute Generalized Advantage Estimation.
    
    Args:
        rewards: List of rewards [r_0, r_1, ...]
        states: List of states [s_0, s_1, ...]
        V: Value function estimates
        gamma: Discount factor
        lambda_: GAE parameter (0=TD, 1=MC)
    
    Returns:
        List of GAE advantage estimates
    """
    T = len(rewards)
    
    # Step 1: Compute all TD errors
    deltas = []
    for t in range(T):
        v_current = V.get(states[t], 0.0)
        v_next = V.get(states[t+1], 0.0) if t+1 < len(states) else 0.0
        delta_t = rewards[t] + gamma * v_next - v_current
        deltas.append(delta_t)
    
    # Step 2: Compute GAE using backward pass
    advantages = [0.0] * T
    advantages[T-1] = deltas[T-1]
    
    for t in range(T-2, -1, -1):
        advantages[t] = deltas[t] + gamma * lambda_ * advantages[t+1]
    
    return advantages, deltas

# Reproduce numerical example
print("=" * 60)
print("Generalized Advantage Estimation (GAE) Numerical Example")
print("=" * 60)

# Trajectory
states = [1, 2, 3, 4]
rewards = [1, 2, 3]
V = {1: 5.0, 2: 4.5, 3: 3.0, 4: 0.0}
gamma = 0.9

print(f"\nTrajectory: s1 --(r=1)--> s2 --(r=2)--> s3 --(r=3)--> s4")
print(f"Value estimates: V(s1)={V[1]}, V(s2)={V[2]}, V(s3)={V[3]}, V(s4)={V[4]}")
print(f"Discount factor: γ={gamma}\n")

# Compute TD errors
_, deltas = compute_gae(rewards, states, V, gamma, lambda_=0.0)
print("Step 1: TD Errors")
print("-" * 40)
for t, delta in enumerate(deltas):
    print(f"δ_{t} = {delta:.2f}")

print("\n" + "="*60)
print("Step 2: GAE with Different λ Values")
print("="*60)

lambda_values = [0.0, 0.5, 0.95, 1.0]
results = {}

for lambda_ in lambda_values:
    advantages, _ = compute_gae(rewards, states, V, gamma, lambda_)
    results[lambda_] = advantages
    
    lambda_label = "0 (TD)" if lambda_ == 0 else ("1 (MC)" if lambda_ == 1 else f"{lambda_}")
    print(f"\nλ = {lambda_label}:")
    for t, adv in enumerate(advantages):
        print(f"  A_{t}^GAE = {adv:.3f}")

# Detailed calculation for λ=0.95 at t=0
print("\n" + "="*60)
print("Detailed Calculation: GAE with λ=0.95 at t=0")
print("="*60)
lambda_ = 0.95
gae_lambda = gamma * lambda_

print(f"\nWorking backwards from t=2 to t=0:")
print(f"\nAt t=2:")
print(f"  A_2 = δ_2 = {deltas[2]:.2f}")

A_2 = deltas[2]
print(f"\nAt t=1:")
print(f"  A_1 = δ_1 + (γλ)A_2")
print(f"      = {deltas[1]:.2f} + ({gamma}×{lambda_})×{A_2:.2f}")
A_1 = deltas[1] + gae_lambda * A_2
print(f"      = {deltas[1]:.2f} + {gae_lambda:.3f}×{A_2:.2f}")
print(f"      = {A_1:.3f}")

print(f"\nAt t=0:")
print(f"  A_0 = δ_0 + (γλ)A_1")
print(f"      = {deltas[0]:.2f} + ({gamma}×{lambda_})×{A_1:.3f}")
A_0 = deltas[0] + gae_lambda * A_1
print(f"      = {deltas[0]:.2f} + {gae_lambda:.3f}×{A_1:.3f}")
print(f"      = {A_0:.3f}")

print(f"\n✓ This matches our GAE computation: A_0 = {results[0.95][0]:.3f}")

### Visualizing GAE for Different λ

In [ ]:
# Compute GAE for a range of lambda values
lambda_range = np.linspace(0, 1, 21)
gae_at_t0 = []

for lambda_ in lambda_range:
    advantages, _ = compute_gae(rewards, states, V, gamma, lambda_)
    gae_at_t0.append(advantages[0])

# Plot
plt.figure(figsize=(10, 6))
plt.plot(lambda_range, gae_at_t0, 'b-', linewidth=3)
plt.scatter([0, 1], [gae_at_t0[0], gae_at_t0[-1]], 
           color='red', s=100, zorder=5, label='TD(0) and MC')
plt.axhline(y=0, color='black', linestyle='--', alpha=0.3, linewidth=1)

plt.xlabel('λ (lambda)', fontsize=12)
plt.ylabel('GAE Advantage at t=0', fontsize=12)
plt.title('How GAE Changes with λ', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)

# Annotate key points
plt.annotate('TD(0)\n(high bias)', xy=(0, gae_at_t0[0]), 
            xytext=(0.1, gae_at_t0[0]-0.05),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')
plt.annotate('Monte Carlo\n(high variance)', xy=(1, gae_at_t0[-1]), 
            xytext=(0.8, gae_at_t0[-1]+0.02),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')
plt.annotate('Sweet spot\n(0.9-0.99)', xy=(0.95, gae_at_t0[-2]), 
            xytext=(0.7, gae_at_t0[-2]+0.05),
            arrowprops=dict(arrowstyle='->', color='green'),
            fontsize=10, color='green', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nKey Insights:")
print(f"- At λ=0: A_0 = {gae_at_t0[0]:.3f} (pure TD, only uses immediate reward)")
print(f"- At λ=1: A_0 = {gae_at_t0[-1]:.3f} (pure MC, uses all rewards)")
print(f"- Typical choice: λ ∈ [0.9, 0.99] balances bias and variance")
print(f"- GAE smoothly interpolates between TD and MC!")

## 4.7 Visual Example: GAE in Corridor World

Now let's see GAE in action! We'll visualize how TD errors are combined with exponential weighting to create advantage estimates.


In [ ]:
# GAE Visual Example
print("="*70)
print("GAE VISUAL EXAMPLE: Exponentially-Weighted TD Errors")
print("="*70)

# Use corridor environment
env = CorridorWorld(length=5)
states_gae, rewards_gae = env.generate_episode()

# Value estimates (partially trained)
V_gae = {0: 4.5, 1: 5.5, 2: 7.0, 3: 8.5, 4: 0.0}
gamma = 0.9

print(f"\nTrajectory: {' -> '.join([f's{s}' for s in states_gae])}")
print(f"Rewards: {rewards_gae}")
print(f"Value estimates: {V_gae}\n")

# Compute TD errors
deltas = []
for t in range(len(rewards_gae)):
    v_curr = V_gae[states_gae[t]]
    v_next = V_gae[states_gae[t+1]]
    delta = rewards_gae[t] + gamma * v_next - v_curr
    deltas.append(delta)
    print(f"δ_{t} = {rewards_gae[t]} + {gamma}×{v_next:.1f} - {v_curr:.1f} = {delta:.2f}")

# Visualize TD errors spatially
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Panel 1: TD Errors
ax1 = axes[0]
for i, delta in enumerate(deltas):
    color = plt.cm.RdYlGn(0.5 + 0.5 * np.tanh(delta))
    rect = Rectangle((i, 0), 1, 1, linewidth=2, 
                    edgecolor='black', facecolor=color)
    ax1.add_patch(rect)
    
    ax1.text(i + 0.5, 0.5, f'δ{i}\n{delta:.2f}', 
           ha='center', va='center', fontsize=11, fontweight='bold')
    
    label = f's{i}'
    ax1.text(i + 0.5, -0.25, label, ha='center', va='top', fontsize=9)

ax1.set_xlim(-0.5, len(deltas) + 0.5)
ax1.set_ylim(-0.5, 1.5)
ax1.set_aspect('equal')
ax1.axis('off')
ax1.set_title('Step 1: TD Errors at Each State', fontsize=13, fontweight='bold')

# Panel 2: GAE with different lambda values
lambda_values = [0.0, 0.5, 0.95]
colors_lambda = ['red', 'orange', 'green']

ax2 = axes[1]
for lambda_idx, lambda_val in enumerate(lambda_values):
    advantages_gae, _ = compute_gae(rewards_gae, states_gae, V_gae, gamma, lambda_val)
    
    y_pos = 0.7 - lambda_idx * 0.25
    for i, adv in enumerate(advantages_gae):
        ax2.text(i + 0.5, y_pos, f'{adv:.2f}', 
               ha='center', va='center', fontsize=10,
               bbox=dict(boxstyle='round', facecolor=colors_lambda[lambda_idx], alpha=0.3))
    
    ax2.text(-0.8, y_pos, f'λ={lambda_val}', 
           ha='right', va='center', fontsize=11, fontweight='bold',
           color=colors_lambda[lambda_idx])

for i in range(len(deltas)):
    ax2.axvline(i + 0.5, color='gray', linestyle='--', alpha=0.3, linewidth=1)
    label = f's{i}'
    ax2.text(i + 0.5, -0.05, label, ha='center', va='top', fontsize=9)

ax2.set_xlim(-1.2, len(deltas) + 0.5)
ax2.set_ylim(-0.1, 0.9)
ax2.axis('off')
ax2.set_title('Step 2: GAE Advantages for Different λ Values', fontsize=13, fontweight='bold')

# Panel 3: Exponential weighting visualization for λ=0.95
ax3 = axes[2]
lambda_viz = 0.95
advantages_viz, _ = compute_gae(rewards_gae, states_gae, V_gae, gamma, lambda_viz)

# Show the weighting for state 0
t = 0
weights = [(gamma * lambda_viz) ** k for k in range(len(deltas))]

# Normalize for visualization
max_weight = max(weights)
for i in range(len(deltas)):
    height = weights[i] / max_weight
    bar_color = plt.cm.Blues(0.3 + 0.7 * height)
    rect = Rectangle((i, 0), 1, height, linewidth=2, 
                    edgecolor='black', facecolor=bar_color)
    ax3.add_patch(rect)
    
    ax3.text(i + 0.5, height + 0.05, f'{weights[i]:.2f}', 
           ha='center', va='bottom', fontsize=9)
    ax3.text(i + 0.5, height/2, f'δ{i}', 
           ha='center', va='center', fontsize=10, fontweight='bold', color='white')
    
    label = f's{i}'
    ax3.text(i + 0.5, -0.08, label, ha='center', va='top', fontsize=9)

ax3.set_xlim(-0.5, len(deltas) + 0.5)
ax3.set_ylim(-0.12, 1.2)
ax3.axis('off')
ax3.set_title(f'Step 3: Exponential Weights (λ={lambda_viz}) for A_0^GAE', 
             fontsize=13, fontweight='bold')
ax3.text(len(deltas)/2, -0.25, 
         f'A_0^GAE = {advantages_viz[0]:.2f} = Σ (γλ)^k δ_(0+k)',
         ha='center', va='top', fontsize=12, 
         bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

plt.suptitle('GAE: Combining TD Errors with Exponential Weighting', 
             fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# Detailed breakdown
print("\n" + "="*70)
print("GAE Computation Breakdown (λ=0.95)")
print("="*70)
lambda_detail = 0.95
advantages_detail, _ = compute_gae(rewards_gae, states_gae, V_gae, gamma, lambda_detail)

for t in range(len(deltas)):
    print(f"\nAdvantage at state s{t}:")
    print(f"  A_{t}^GAE = ", end="")
    terms = []
    for k in range(len(deltas) - t):
        weight = (gamma * lambda_detail) ** k
        delta_val = deltas[t + k]
        terms.append(f"{weight:.3f}×{delta_val:.2f}")
    print(" + ".join(terms))
    print(f"  A_{t}^GAE = {advantages_detail[t]:.3f}")

print("\n" + "="*70)
print("Key Insights:")
print("="*70)
print(f"1. λ=0: Pure TD (only immediate δ_t)")
print(f"2. λ=1: Monte Carlo (all future δs equally weighted by γ^k)")
print(f"3. λ=0.95: Exponential decay (near future matters more than far future)")
print(f"4. GAE smoothly interpolates between TD and MC!")


---

## 4.7 Putting It All Together: GAE in Policy Gradient

Now let's see how GAE is used in practice for policy gradient algorithms.

### The Policy Gradient with GAE

Recall the policy gradient theorem:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) A^\pi(s_t, a_t)\right]$$

In practice, we:
1. **Collect trajectories** using current policy $\pi_\theta$
2. **Estimate advantages** using GAE: $\hat{A}_t^{GAE}$
3. **Update policy** in the direction of:
   $$\nabla_\theta \log \pi_\theta(a_t|s_t) \hat{A}_t^{GAE}$$

### Complete Algorithm

```
Initialize:
- Policy network π_θ
- Value network V_φ
- Hyperparameters: γ, λ, learning rates

Loop:
  1. Collect trajectories using π_θ
     For each trajectory:
       - States: s_0, s_1, ..., s_T
       - Actions: a_0, a_1, ..., a_{T-1}
       - Rewards: r_0, r_1, ..., r_{T-1}
  
  2. Compute advantages using GAE:
     For each timestep t:
       a) Compute TD error: δ_t = r_t + γV(s_{t+1}) - V(s_t)
       b) Compute GAE: A_t = Σ_{k=0}^∞ (γλ)^k δ_{t+k}
  
  3. Update value network V_φ:
     Target: G_t^λ = A_t + V(s_t)
     Loss: (V(s_t) - G_t^λ)²
  
  4. Update policy network π_θ:
     Gradient: ∇_θ log π_θ(a_t|s_t) A_t
```

In [ ]:
def policy_gradient_with_gae_step(trajectory: dict, 
                                 V: dict,
                                 gamma: float,
                                 lambda_: float) -> dict:
    """
    Compute quantities needed for a policy gradient update with GAE.
    
    Args:
        trajectory: Dict with 'states', 'actions', 'rewards'
        V: Value function
        gamma: Discount factor
        lambda_: GAE parameter
    
    Returns:
        Dict with advantages, value targets, and other info
    """
    states = trajectory['states']
    rewards = trajectory['rewards']
    
    # Compute GAE advantages
    advantages, deltas = compute_gae(rewards, states, V, gamma, lambda_)
    
    # Compute value targets (for training value function)
    # Target = advantage + baseline
    value_targets = [advantages[t] + V.get(states[t], 0.0) 
                    for t in range(len(rewards))]
    
    return {
        'advantages': advantages,
        'value_targets': value_targets,
        'td_errors': deltas
    }

# Example: Process a trajectory
print("=" * 60)
print("Policy Gradient Update with GAE")
print("=" * 60)

trajectory = {
    'states': [1, 2, 3, 4],
    'actions': ['right', 'right', 'right'],  # just for illustration
    'rewards': [1, 2, 3]
}

V = {1: 5.0, 2: 4.5, 3: 3.0, 4: 0.0}
gamma = 0.9
lambda_ = 0.95

update_data = policy_gradient_with_gae_step(trajectory, V, gamma, lambda_)

print(f"\nTrajectory processed with γ={gamma}, λ={lambda_}")
print("\n" + "-"*60)
print(f"{'Time':<6} {'State':<8} {'Action':<10} {'Advantage':<12} {'Value Target':<12}")
print("-"*60)

for t in range(len(trajectory['rewards'])):
    state = trajectory['states'][t]
    action = trajectory['actions'][t]
    adv = update_data['advantages'][t]
    target = update_data['value_targets'][t]
    print(f"t={t:<4} s{state:<7} {action:<10} {adv:>8.3f}     {target:>8.3f}")

print("\n" + "="*60)
print("How These Are Used:")
print("="*60)
print("")
print("1. POLICY UPDATE:")
print("   For each timestep t:")
print("   - Compute: ∇_θ log π_θ(a_t|s_t)")
print("   - Weight by: A_t (the advantage)")
print("   - Update: θ ← θ + α × ∇_θ log π_θ(a_t|s_t) × A_t")
print("")
print("2. VALUE FUNCTION UPDATE:")
print("   For each timestep t:")
print("   - Target: value_target_t")
print("   - Loss: (V(s_t) - value_target_t)²")
print("   - Update value network to minimize loss")
print("")
print("Advantages are:")
if all(a > 0 for a in update_data['advantages']):
    print("  ✓ All positive → all actions were better than expected!")
elif all(a < 0 for a in update_data['advantages']):
    print("  ✗ All negative → all actions were worse than expected")
else:
    print("  ○ Mixed → some actions good, some bad")

---

# 5. Summary and Key Takeaways

Let's recap what we've learned!

## 5.1 Core Concepts

### TD Learning
- **Update online** without waiting for episode to end
- **Bootstrap**: Use our own estimates to improve our estimates
- **TD error**: $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$
- **Tradeoff**: Low variance but biased (initially)

### n-step Returns
- **Middle ground** between TD (1-step) and Monte Carlo (∞-step)
- Use $n$ real rewards, then bootstrap
- **Formula**: $G_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n})$
- **Tuning**: Larger $n$ → less bias, more variance

### Advantage Functions
- **Measure**: How much better is action $a$ than average?
- **Definition**: $A(s,a) = Q(s,a) - V(s)$
- **Why**: Centers updates around zero, reduces variance
- **Estimation**: Can use TD errors, n-step returns, or GAE

### GAE (Generalized Advantage Estimation)
- **Exponentially-weighted average** of all n-step advantages
- **Formula**: $\hat{A}_t^{GAE} = \sum_{k=0}^{\infty} (\gamma\lambda)^k \delta_{t+k}$
- **Parameter $\lambda$**: Controls bias-variance tradeoff
  - $\lambda=0$: Pure TD (high bias, low variance)
  - $\lambda=1$: Pure MC (no bias, high variance)
  - $\lambda \in [0.9, 0.99]$: Practical sweet spot

## 5.2 The Bias-Variance Tradeoff

All these methods trade off bias and variance:

| Method | Bias | Variance | When to Use |
|--------|------|----------|-------------|
| 1-step TD | High (if V is wrong) | Low | Fast updates, short episodes |
| n-step | Medium | Medium | Balance for specific problems |
| Monte Carlo | None | High | Long episodes, accurate returns |
| GAE (λ=0.95) | Low | Low | Modern policy gradient (PPO, etc.) |

## 5.3 Practical Usage

In modern RL (e.g., PPO, A3C):
1. Use **GAE** with $\lambda \in [0.9, 0.99]$
2. Train a **value network** $V_\phi(s)$ alongside policy
3. Collect trajectories and compute GAE advantages
4. Update both policy and value function

## 5.4 Key Equations to Remember

**TD Error**:
$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

**n-step Return**:
$$G_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n})$$

**Advantage**:
$$A(s,a) = Q(s,a) - V(s)$$

**GAE**:
$$\hat{A}_t^{GAE} = \sum_{k=0}^{\infty} (\gamma\lambda)^k \delta_{t+k}$$

**Recursive GAE** (for implementation):
$$\hat{A}_t = \delta_t + \gamma\lambda \hat{A}_{t+1}$$

## 5.5 Next Steps

Now that you understand these concepts, you're ready to:
- Implement **Proximal Policy Optimization (PPO)**
- Understand **Actor-Critic** methods
- Study **Trust Region Policy Optimization (TRPO)**
- Explore other advanced policy gradient methods

**Remember**: All modern policy gradient algorithms use some form of advantage estimation. GAE is the most popular choice because it gives us a simple parameter $\lambda$ to tune the bias-variance tradeoff!

---

## Congratulations!

You now have a deep understanding of TD learning, n-step returns, advantages, and GAE. These are fundamental building blocks of modern deep reinforcement learning. Keep practicing and experimenting with these concepts!

### Additional Resources

- **Original GAE Paper**: [High-Dimensional Continuous Control Using Generalized Advantage Estimation](https://arxiv.org/abs/1506.02438)
- **Sutton & Barto**: Reinforcement Learning: An Introduction (Chapters 6, 7, 12)
- **Spinning Up in Deep RL**: https://spinningup.openai.com/